In [ ]:
  !unzip /content/bep.zip

Archive:  /content/bep.zip
   creating: bep dataset/
  inflating: bep dataset/JONATHAN KUFINGATE.pdf  
  inflating: bep dataset/MANTHAN OJHA RES.pdf  
  inflating: bep dataset/NLP_RESUME.pdf  
  inflating: bep dataset/JAVED s.pdf  


In [ ]:
# ------------------------
# 1. Imports
# ------------------------
!pip install pdfminer.six sentence-transformers spacy pandas tqdm

import os, re, string
import pandas as pd
from tqdm import tqdm
import spacy
from sentence_transformers import SentenceTransformer, util
from pdfminer.high_level import extract_text

# Load models once
nlp = spacy.load("en_core_web_sm")
model = SentenceTransformer('all-MiniLM-L6-v2')

# ------------------------
# 2. Read PDF text
# ------------------------
def extract_resume_text(pdf_path):
    """Extract text from a PDF resume"""
    return extract_text(pdf_path)

# ------------------------
# 3. Preprocess text
# ------------------------
def preprocess_text(text):
    """Lowercase, remove punctuation, stopwords, and lemmatize"""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    return " ".join(tokens)

# ------------------------
# 4. Keyphrase score
# ------------------------
def keyphrase_score(text, skills):
    """Calculate ratio of key skills present in the text"""
    hits = sum(1 for s in skills if s.lower() in text)
    return hits / len(skills) if skills else 0

# ------------------------
# 5. Semantic similarity score
# ------------------------
def semantic_score(resume_text, job_description):
    """Compute semantic similarity between resume and job description"""
    emb_resume = model.encode(resume_text, convert_to_tensor=True)
    emb_job = model.encode(job_description, convert_to_tensor=True)
    sim = util.pytorch_cos_sim(emb_resume, emb_job)
    return float(sim)

# ------------------------
# 6. Combined score
# ------------------------
def combined_score(resume_path, job_description, skills):
    raw_text = extract_resume_text(resume_path)
    clean_resume = preprocess_text(raw_text)
    k_score = keyphrase_score(clean_resume, skills)
    s_score = semantic_score(clean_resume, preprocess_text(job_description))
    final = 0.4 * k_score + 0.6 * s_score
    return final, k_score, s_score, raw_text

# ------------------------
# 7. Information extraction
# ------------------------
def extract_emails(text):
    return re.findall(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', text)
def extract_phone_numbers(text):
    return re.findall(r'(\+?\d[\d \-\(\)]{9,}\d)', text)

def extract_name(text):
    doc = nlp(text)
    names = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    return names[0] if names else None

# ------------------------
# 8. Batch scanning multiple resumes
# ------------------------
def analyze_resumes(folder_path, job_description, skills):
    results = []
    pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".pdf")]

    for file in tqdm(pdf_files, desc="Analyzing resumes", colour="cyan"):
        pdf_path = os.path.join(folder_path, file)
        try:
            final, k, s, raw_text = combined_score(pdf_path, job_description, skills)
            name = extract_name(raw_text)
            emails = extract_emails(raw_text)
            phones = extract_phone_numbers(raw_text)

            results.append({
                "Filename": file,
                "Name": name,
                "Emails": ", ".join(emails) if emails else "",
                "Phones": ", ".join(phones) if phones else "",
                "Keyphrase_Score": round(k, 3),
                "Semantic_Score": round(s, 3),
                "Final_Score": round(final, 3)
            })
        except Exception as e:
            print(f"⚠️ Error processing {file}: {e}")

    df = pd.DataFrame(results)
    if not df.empty:
        df.sort_values(by="Final_Score", ascending=False, inplace=True)
        df.reset_index(drop=True, inplace=True)
    return df

# ------------------------
# 9. Example usage
# ------------------------
if __name__ == "__main__":
    folder = "/content/bep dataset"  # 📂 folder containing multiple resumes
    job_desc = """We are looking for a Python developer with NLP and SQL experience."""
    required_skills = ["python", "sql", "natural language processing", "django"]

    df_results = analyze_resumes(folder, job_desc, required_skills)

    print("\n=== 📊 Resume Matching Results ===")
    display(df_results)

    # Save to Excel
    save_path = "/content/resume_match_results.xlsx"
    df_results.to_excel(save_path, index=False)
    print(f"\n✅ Results saved to {save_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 103.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Analyzing resumes: 100%|██████████| 4/4 [00:03<00:00,  1.31it/s]


=== 📊 Resume Matching Results ===


,Filename,Name,Emails,Phones,Keyphrase_Score,Semantic_Score,Final_Score
0,JONATHAN KUFINGATE.pdf,JONATHAN KUFINGATE,,,0.5,0.353,0.412
1,NLP_RESUME.pdf,ANUVA GOYAL \n\nD.O.B.,anuvagoyal111@gmail.com,+91 9520349542,0.5,0.300,0.380
2,MANTHAN OJHA RES.pdf,MANTHAN OJHA,,,0.5,0.291,0.374
3,JAVED s.pdf,JAVED NUSRANI,-nusranino@gmail.com,,0.0,0.126,0.075



✅ Results saved to /content/resume_match_results.xlsx
